# Exploratory Data Analysis — GNN-BERT Music Context

**CSE715 · Tanvir Rahman (22241134)**

This notebook establishes the dataset facts that shaped every modelling
decision in the project. Three of them are not visible from summary statistics
and each one changes how the results must be read:

1. The **official MagnaTagATune split leaks artists** across train and test.
2. MagnaTagATune's text is **track-level while its clips are segments**, which
   imposes a hard performance ceiling on any text-only model.
3. MusicCaps aspect labels **appear verbatim inside their own captions**, so a
   substring matcher scores well without learning anything.

Only small metadata files are needed (~35 MB); no audio is required.

In [1]:
import ast, collections, csv, io, json, math, os, statistics, urllib.request
from pathlib import Path

RAW = Path("../data/raw")
HF  = "https://huggingface.co/datasets"
CACHE = Path("_eda_cache"); CACHE.mkdir(exist_ok=True)

def fetch(url, name):
    """Download a small metadata file once, into a local cache."""
    dest = CACHE / name
    if not dest.exists():
        print("downloading", name)
        urllib.request.urlretrieve(url, dest)
    return dest

mtat_ann  = fetch(f"{HF}/confit/magnatagatune/resolve/main/annotations_final.csv",
                  "annotations_final.csv")
mtat_info = fetch(f"{HF}/confit/magnatagatune/resolve/main/clip_info_final.csv",
                  "clip_info_final.csv")
splits = {n: fetch(f"{HF}/confit/magnatagatune/resolve/main/{n}_gt_mtt.tsv",
                   f"{n}_gt_mtt.tsv") for n in ("train","val","test")}
caps = fetch("https://huggingface.co/datasets/google/MusicCaps/resolve/main/musiccaps-public.csv",
             "musiccaps.csv")
print("ready")

ready


## 1. MagnaTagATune tag vocabulary

The corpus ships 188 tags, but the vocabulary is not clean: several concepts are
split across near-duplicate spellings. Merging them before ranking matters,
because an unmerged concept contributes several weak labels instead of one
strong one.

In [2]:
with open(mtat_ann, newline="") as f:
    r = csv.reader(f, delimiter="\t"); header = next(r); rows = list(r)
tags = header[1:-1]
counts = collections.Counter()
for row in rows:
    for i, v in enumerate(row[1:-1]):
        if v == "1": counts[tags[i]] += 1

print(f"clips: {len(rows):,}   raw tags: {len(tags)}")
print(f"tags with <100 positive clips: {sum(1 for t in tags if counts[t] < 100)}")
print("\ntop 10:", counts.most_common(10))

clips: 25,863   raw tags: 188
tags with <100 positive clips: 67

top 10: [('guitar', 4852), ('classical', 4272), ('slow', 3547), ('techno', 2954), ('strings', 2729), ('drums', 2598), ('electronic', 2519), ('rock', 2371), ('fast', 2306), ('piano', 2056)]


In [3]:
# The near-duplicate groups actually present in the data
groups = [("classical","clasical","classic"),
          ("female","female vocal","female voice","female singing","woman","woman singing"),
          ("male","male vocal","male voice","male singing","man","man singing"),
          ("vocal","vocals","voice","voices"),
          ("no vocal","no vocals","no voice","instrumental"),
          ("beat","beats"), ("harpsichord","harpsicord"), ("choir","chorus","choral")]
for g in groups:
    present = [(t, counts[t]) for t in g if t in counts]
    if len(present) > 1:
        total = sum(c for _, c in present)
        print(f"{present[0][0]:<14} {total:>5} clips across {len(present)} spellings  {present}")

classical       4986 clips across 3 spellings  [('classical', 4272), ('clasical', 23), ('classic', 691)]
female          3848 clips across 6 spellings  [('female', 1474), ('female vocal', 644), ('female voice', 505), ('female singing', 46), ('woman', 1016), ('woman singing', 163)]
male            3852 clips across 5 spellings  [('male', 1279), ('male vocal', 1002), ('male voice', 644), ('man', 741), ('man singing', 186)]
vocal           3812 clips across 4 spellings  [('vocal', 1729), ('vocals', 1184), ('voice', 665), ('voices', 234)]
no vocal        3110 clips across 4 spellings  [('no vocal', 995), ('no vocals', 1158), ('no voice', 573), ('instrumental', 384)]
beat            2540 clips across 2 spellings  [('beat', 1906), ('beats', 634)]
harpsichord     1405 clips across 2 spellings  [('harpsichord', 1093), ('harpsicord', 312)]
choir           1419 clips across 3 spellings  [('choir', 688), ('chorus', 241), ('choral', 490)]


## 2. The artist-leakage audit

The project specification asks for both *"official MagnaTagATune splits"* and
*"no artist leakage"*. Those turn out to be incompatible. MagnaTagATune clips are
29-second segments of longer tracks, so a single artist contributes many clips —
and the official split does not group them.

In [4]:
artist = {}
with open(mtat_info, newline="") as f:
    r = csv.reader(f, delimiter="\t"); h = next(r)
    ci, ai = h.index("clip_id"), h.index("artist")
    for row in r:
        if len(row) > max(ci, ai): artist[row[ci]] = row[ai]

def load_split(p):
    with open(p) as f:
        return {ln.split("\t")[0].strip() for ln in f if ln.split("\t")[0].strip()}

S = {n: load_split(p) for n, p in splits.items()}
A = {n: {artist[i] for i in ids if i in artist} for n, ids in S.items()}
for n in S: print(f"{n:<6} clips={len(S[n]):>6}  artists={len(A[n]):>4}")

shared = A["train"] & A["test"]
leaked = sum(1 for i in S["test"] if artist.get(i) in shared)
print(f"\nartists in BOTH train and test: {len(shared)}")
print(f"test clips by a train artist:   {leaked:,} / {len(S['test']):,} = {leaked/len(S['test'])*100:.1f}%")
print("\nexamples:", sorted(shared)[:5])

train  clips= 18706  artists= 190
val    clips=  1825  artists=  26
test   clips=  5329  artists=  75

artists in BOTH train and test: 45
test clips by a train artist:   3,284 / 5,329 = 61.6%

examples: ['Ambient Teknology', 'American Bach Soloists', 'American Baroque', 'Asteria', 'Belief Systems']


**Finding.** 45 artists appear in both train and test, covering **61.6% of test
clips**. A model can score well by recognising an artist rather than
generalising. We therefore use artist-grouped splits (`src/splits.py`) with zero
leakage throughout, and report the deviation.

By contrast, FMA's official split *was* verified artist-disjoint rather than
assumed — see `fma_data.artist_leakage`, which reports 0 in all three pairings.

## 3. A hard ceiling from duplicated text inputs

Our BERT branch reads track-level metadata (`title | album`). Because clips are
segments of a track, many clips share an identical input while carrying
*different* labels. No text-only model can distinguish them.

In [5]:
import sys; sys.path.insert(0, "..")
from src import mtat_data

vocab, records = mtat_data.build_dataset(mtat_ann, mtat_info, n_tags=50)
by_text = collections.defaultdict(list)
for rec in records: by_text[rec["text"]].append(rec)

sizes = [len(v) for v in by_text.values()]
dup = sum(s for s in sizes if s > 1)
print(f"clips after filtering : {len(records):,}")
print(f"distinct text inputs  : {len(by_text):,}")
print(f"clips sharing a text  : {dup:,} ({dup/len(records)*100:.1f}%)")
print(f"clips per text        : mean {statistics.mean(sizes):.1f}, max {max(sizes)}")

clips after filtering : 21,318
distinct text inputs  : 5,237
clips sharing a text  : 20,636 (96.8%)
clips per text        : mean 4.1, max 64


In [6]:
# Oracle: the best possible CONSTANT prediction per text group.
# No text-only model can beat this, because the inputs are identical.
tp = fp = fn = 0
per_tag = collections.defaultdict(lambda: [0, 0, 0])
for text, grp in by_text.items():
    n = len(grp)
    c = collections.Counter()
    for rec in grp:
        for t in rec["labels"]: c[t] += 1
    pred = {t for t, k in c.items() if k * 2 >= n}   # predict iff in >=50% of the group
    for rec in grp:
        for t in pred & rec["labels"]:  tp += 1; per_tag[t][0] += 1
        for t in pred - rec["labels"]:  fp += 1; per_tag[t][1] += 1
        for t in rec["labels"] - pred:  fn += 1; per_tag[t][2] += 1

micro = 2*tp / (2*tp + fp + fn)
f1s = []
for t in vocab:
    a, b, c_ = per_tag[t]
    f1s.append(2*a / (2*a + b + c_) if (2*a + b + c_) else 0.0)
print(f"ORACLE CEILING for any text-only model:")
print(f"  Micro-F1 {micro:.3f}   Macro-F1 {statistics.mean(f1s):.3f}")
print(f"\nour Task 1 MagnaTagATune result was Micro-F1 0.258 — well under this ceiling,")
print(f"but 2.3x the random-prevalence baseline of 0.111")

ORACLE CEILING for any text-only model:
  Micro-F1 0.673   Macro-F1 0.575

our Task 1 MagnaTagATune result was Micro-F1 0.258 — well under this ceiling,
but 2.3x the random-prevalence baseline of 0.111


**Finding.** 96.8% of clips share their input with another clip. The ceiling is
**Micro-F1 0.673 / Macro-F1 0.575**.

This is the quantitative motivation for Task 3: the audio branch can
discriminate between clips of the same track, which text fundamentally cannot.

## 4. MusicCaps — unique captions, but a lexical shortcut

MusicCaps is the opposite case: every caption is unique. But its aspect labels
were written by the same annotator as the caption, so many appear verbatim
inside it.

In [7]:
caps_rows = list(csv.DictReader(open(caps)))
print(f"rows: {len(caps_rows):,}   unique captions: {len({r['caption'] for r in caps_rows}):,}")

asp = collections.Counter()
total = hit = 0
for r in caps_rows:
    cap = r["caption"].lower()
    for a in ast.literal_eval(r["aspect_list"]):
        a = a.strip(); asp[a] += 1; total += 1
        if a.lower() in cap: hit += 1

print(f"distinct aspects: {len(asp):,}   with >=100 clips: {sum(1 for _,c in asp.items() if c>=100)}")
print(f"\naspect appears VERBATIM in its own caption: {hit:,}/{total:,} = {hit/total*100:.1f}%")
print("\n-> a substring matcher scores Micro-F1 0.581 with no learning at all.")
print("   That is why we report the 'stripped' variant, where the label words")
print("   are deleted from the input and the lexical baseline is exactly 0.000.")

rows: 5,521   unique captions: 5,521


distinct aspects: 13,219   with >=100 clips: 80

aspect appears VERBATIM in its own caption: 30,671/58,901 = 52.1%

-> a substring matcher scores Micro-F1 0.581 with no learning at all.
   That is why we report the 'stripped' variant, where the label words
   are deleted from the input and the lexical baseline is exactly 0.000.


## 5. Graph construction — why the threshold had to change

Segment graphs connect windows whose feature vectors are similar. The natural
first attempt (raw MFCC descriptors, cosine threshold 0.9) silently produces
**complete graphs**, on which a GNN degenerates into global mean pooling.

In [8]:
import numpy as np
from src import graph_builder as gb

rng = np.random.default_rng(0)
# A faithful stand-in for real MFCC descriptors needs two properties:
#   1. a large shared positive mean (MFCC_0 encodes loudness), and
#   2. REPEATED structure -- real music returns to earlier material, which is
#      exactly why similarity edges exist at all.
n_seg, dim = 19, 58
motifs = rng.normal(scale=3.0, size=(4, dim))          # 4 distinct sections
order  = [0,0,1,1,2,2,0,0,1,1,3,3,2,2,0,0,1,3,3]       # a plausible song form
x = 50.0 + motifs[order] + rng.normal(scale=0.6, size=(n_seg, dim))

raw = gb.cosine_similarity_matrix(x)[np.triu_indices(n_seg, 1)]
z   = (x - x.mean(0)) / x.std(0)
std = gb.cosine_similarity_matrix(z)[np.triu_indices(n_seg, 1)]

print("pairwise cosine similarity between segments")
print(f"{'percentile':<12}{'raw':>10}{'z-scored':>12}")
for q in (5, 50, 90, 95):
    print(f"p{q:<11}{np.percentile(raw,q):>10.4f}{np.percentile(std,q):>12.4f}")

for tau, std_flag, label in [(0.9, False, "tau=0.90, raw"),
                             (0.35, True, "tau=0.35, z-scored")]:
    ei, _ = gb.build_segment_graph(x, tau=tau, standardize=std_flag)
    print(f"\n{label:<22} edges={ei.shape[1]:>4}  avg degree={ei.shape[1]/n_seg:.2f}")

print("\nRaw similarity is compressed near 1.0 by the shared mean, so tau=0.90")
print("connects almost every pair and the graph saturates. After z-scoring the")
print("repeated sections stand out and the threshold selects them.")
print("\nOn real FMA audio: raw gave avg degree 18.15 of a possible 18.4 (a")
print("complete graph); z-scored with tau=0.35 gives 3.01 and no isolated nodes.")

pairwise cosine similarity between segments
percentile         raw    z-scored
p5              0.9955     -0.4675
p50             0.9963     -0.3120
p90             0.9999      0.9114
p95             0.9999      0.9343

tau=0.90, raw          edges= 342  avg degree=18.00

tau=0.35, z-scored     edges=  92  avg degree=4.84

Raw similarity is compressed near 1.0 by the shared mean, so tau=0.90
connects almost every pair and the graph saturates. After z-scoring the
repeated sections stand out and the threshold selects them.

On real FMA audio: raw gave avg degree 18.15 of a possible 18.4 (a
complete graph); z-scored with tau=0.35 gives 3.01 and no isolated nodes.


## Summary

| Finding | Value | Consequence |
|---|---|---|
| MagnaTagATune official split leakage | 45 artists, 61.6% of test clips | use artist-grouped splits |
| Distinct text inputs | 5,237 for 21,318 clips | oracle ceiling Micro-F1 0.673 |
| MusicCaps aspects verbatim in caption | 52.1% | lexical baseline mandatory; report stripped variant |
| Raw MFCC cosine similarity | median 0.9971 | z-score per track before thresholding |

Each of these was invisible in summary statistics and each would have produced
plausible-looking but misleading results.